In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from dpu_utils.utils import RichPath
from pathlib import Path

In [3]:
import warnings

from tqdm import TqdmWarning
warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/Cellar/python@3.9/3.9.19_1/Frameworks/Python.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/local/Cellar/python@3.9/3.9.19_1/Frameworks/Python.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/germanarutyunov/DataspellProjects/graph-typer/venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  Fil

In [17]:
resolved_path = Path("~/git-py/tensorised-data/train").expanduser()

In [18]:
train_data_path = RichPath.create(str(resolved_path))

In [19]:
train_data = train_data_path.iterate_filtered_files_in_dir('chunk_*.pkl.gz')

In [3]:
# Add the project root to your Python path
import sys
sys.path.append('../')

In [21]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

In [22]:
from dpu_utils.utils import ChunkWriter
from graph_coder.data.wrapper import preprocess_item

In [23]:
output_path = Path("~/git-py/processed-data/train").expanduser()
output_path.mkdir(parents=True, exist_ok=True)

_necessary_keys = ['cg_node_label_token_ids', 'cg_edges', 'target_node_idxs', 'variable_target_class']

In [24]:
i = 0

batch = []

for file_path in iter(train_data):
    for data_chunk in iter(file_path.read_by_file_suffix()):
        # check for necessary keys, skip if empty
        if not all(k in data_chunk for k in _necessary_keys):
            continue

        if len(batch) == 2:
            break

        n_edges = 0

        for edges in data_chunk['cg_edges']:
            n_edges += edges.shape[0]

        edge_index = torch.zeros((2, n_edges), dtype=torch.long)
        edge_attr = torch.zeros((n_edges, 1), dtype=torch.long)

        start_offset = 0

        for i, edges in enumerate(data_chunk['cg_edges']):
            if edges.shape[0] > 0:  # Make sure there are edges of this type
                edge_index[:, start_offset:start_offset + edges.shape[0]] = torch.tensor(edges, dtype=torch.long).t()
                edge_attr[start_offset:start_offset + edges.shape[0], 0] = i
                start_offset += edges.shape[0]

        # Create node labels tensor
        y = torch.full((data_chunk['cg_node_label_token_ids'].shape[0],), -100)
        idx = torch.tensor(data_chunk['target_node_idxs'], dtype=torch.long)
        y[idx] = torch.tensor(data_chunk['variable_target_class'], dtype=torch.long)

        # Create dict
        data = dict(
            idx=torch.LongTensor([i]),
            x=torch.tensor(data_chunk['cg_node_label_token_ids'], dtype=torch.long),
            y=y,
            edge_index=edge_index,
            edge_attr=edge_attr,
        )

        data = preprocess_item(data, mask=False)

        # Add to the chunk writer
        batch.append(data)
        i += 1

In [27]:
from graph_coder.data.collator import collator

In [28]:
batched_data = collator(batch)

In [29]:
for k, v in batched_data.items():
    print(k, v.size())

idx torch.Size([2, 1])
edge_index torch.Size([2, 2356])
edge_data torch.Size([2356, 1])
node_data torch.Size([1166, 1])
in_degree torch.Size([1166])
out_degree torch.Size([1166])
lap_eigvec torch.Size([1166, 784])
lap_eigval torch.Size([1166, 784])
y torch.Size([2, 2371])
node_num torch.Size([2, 1])
edge_num torch.Size([2, 1])


In [30]:
from graph_coder.tasks.deep_similarity_learning import DeepSimilarityLearningTask
warnings.filterwarnings("ignore", "A module that was compiled")

In [31]:
from argparse import ArgumentParser

parser = ArgumentParser()

In [32]:

DeepSimilarityLearningTask.add_args(parser)

In [33]:
args = parser.parse_args([])

In [34]:
# print args
for k, v in vars(args).items():
    print(k, v)

dataset_name typilus
num_classes 100
max_nodes 10000
dataset_source None
num_atoms 10514
num_edges 10
num_in_degree 512
num_out_degree 512
num_spatial 512
num_edge_dis 128
multi_hop_max_dist 5
spatial_pos_max 1024
edge_type multi_hop
pretrained_model_name none
load_pretrained_model_output_layer False
train_epoch_shuffle True
user_data_dir 
dataset_root ~/data
processed_dir processed-dir
num_data_workers 4


In [35]:
from tokengt.models import TokenGTModel

In [36]:
from tokengt.models.tokengt import tokengt_mini_architecture

In [37]:
tokengt_mini_architecture(args)

In [38]:
model = TokenGTModel.build_model(args, None)

2025-03-16 09:26:16 | INFO | tokengt.models.tokengt | Namespace(dataset_name='typilus', num_classes=100, max_nodes=10000, dataset_source=None, num_atoms=10514, num_edges=10, num_in_degree=512, num_out_degree=512, num_spatial=512, num_edge_dis=128, multi_hop_max_dist=5, spatial_pos_max=1024, edge_type='multi_hop', pretrained_model_name='none', load_pretrained_model_output_layer=False, train_epoch_shuffle=True, user_data_dir='', dataset_root='~/data', processed_dir='processed-dir', num_data_workers=4, encoder_embed_dim=256, encoder_layers=1, encoder_attention_heads=4, encoder_ffn_embed_dim=256, dropout=0.1, attention_dropout=0.1, act_dropout=0.0, activation_fn='gelu', encoder_normalize_before=True, apply_graphormer_init=True, share_encoder_input_output_embed=False, prenorm=True, postnorm=False, rand_node_id=False, rand_node_id_dim=64, orf_node_id=False, orf_node_id_dim=64, lap_node_id=False, lap_node_id_k=8, lap_node_id_sign_flip=False, lap_node_id_eig_dropout=0.0, type_id=True, stochast

In [39]:
target_representations = model(batched_data)

In [40]:
import torch.nn.functional as F

In [41]:
labels = batched_data['y']

In [42]:
target_logits = model.encoder.embed_out(target_representations) + model.encoder.lm_output_learned_bias
target_probs = F.softmax(target_logits, dim=-1)

class_loss = F.cross_entropy(
    target_probs.view(-1, target_probs.size(-1)),
    labels.view(-1),
    reduction='mean'
)

In [43]:
has_value = labels != -100

In [44]:
labels_with_value = labels[has_value]
representations_with_value = target_representations[has_value]

In [45]:
a = labels_with_value[:, None]
b = labels_with_value[None, :]

typed_annotation_pairs_are_equal = a == b

In [52]:
distances = torch.cdist(representations_with_value, representations_with_value, p=1)

In [53]:
margin = 1.0

In [54]:

max_positive_distance = torch.max(distances * typed_annotation_pairs_are_equal, dim=-1)[0]


In [55]:
neg_dist_filter = distances <= (max_positive_distance.unsqueeze(-1) + margin)

In [56]:
pos_mask = typed_annotation_pairs_are_equal + torch.eye(distances.size(0))
neg_dist_filter = neg_dist_filter.float() * (1 - pos_mask)
mean_negative_distances = torch.sum(distances * neg_dist_filter, dim=-1) / (torch.sum(neg_dist_filter, dim=-1) + 1e-10)

min_negative_distance = torch.min(distances + pos_mask * 3000, dim=-1)[0]
pos_dist_filter = (distances >= (min_negative_distance.unsqueeze(-1) - margin)).float() * typed_annotation_pairs_are_equal
mean_positive_distances = torch.sum(distances * pos_dist_filter, dim=-1) / (torch.sum(pos_dist_filter, dim=-1) + 1e-10)

In [57]:
triplet_loss = 0.5 * F.relu(mean_positive_distances - min_negative_distance + margin)
triplet_loss += 0.5 * F.relu(max_positive_distance - mean_negative_distances + margin)

In [59]:
loss = class_loss + triplet_loss.mean()

In [60]:
loss

tensor(35.4763, grad_fn=<AddBackward0>)

In [4]:
from graph_coder.data.typilus_dataset import TypilusDataset

/Users/germanarutyunov/DataspellProjects/graph-typer/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/Cellar/python@3.9/3.9.19_1/Frameworks/Python.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/local/Cellar/python@3.9/3.9.19_1

In [5]:
from graph_coder.tasks.deep_similarity_learning import DeepSimilarityLearningConfig

In [7]:

dataset = TypilusDataset(
    cfg=DeepSimilarityLearningConfig(
        dataset_root="~/git-py",
        processed_dir="processed-data",
        num_classes=100,
    ),
    split='train',
)

Loading /Users/germanarutyunov/git-py/processed-data/train/data_chunk_0.pkl.gz: 100%|██████████| 872/872 [00:00<00:00, 285447.05it/s]
Loading /Users/germanarutyunov/git-py/processed-data/train/data_chunk_2.pkl.gz: 100%|██████████| 117/117 [00:00<00:00, 314572.80it/s]
Loading /Users/germanarutyunov/git-py/processed-data/train/data_chunk_1.pkl.gz: 100%|██████████| 271/271 [00:00<00:00, 118748.06it/s]


In [8]:
dataset[0]

{'x': tensor([  129,    55,    12,     1,  1254,  1254,    38,     7,    19,    39,
            40,    13,   329,     2,    39,    40,    13,   251,     2,    13,
            37,     2,    13,   251,     2,    13, 10001,     2,    13,  1133,
             2,    13,   751,     2,    13,  1960,     2,    13,   107,     2,
            13,   169,     2,    39,    40,    13,  1960,     2,    39,    40,
            13,   329,     2,    39,    40,    13,   251,     2,    39,    40,
            13,    30,     2,    13,    37,     2,    18,    12,     1,     0,
             0,    38,     7,    19,    39,    40,    13,   251,     2,    13,
            37,     2,    13,   251,     2,    13, 10001,     2,    13,  1133,
             2,    13,   751,     2,    13,  1960,     2,    39,    40,    13,
            30,     2,    13,   169,     2,    39,    40,    13,    30,     2,
            39,    40,    13,   329,     2,    39,    40,    13,   251,     2,
            39,    40,    13,    30,     2,    